In [1]:
print("Hello world")

Hello world


In [2]:
!! pip install seaborn

['Requirement already satisfied: seaborn in /home/ubuntu/.pyenv/versions/3.11.13/lib/python3.11/site-packages (0.13.2)',
 'Requirement already satisfied: numpy!=1.24.0,>=1.20 in /home/ubuntu/.pyenv/versions/3.11.13/lib/python3.11/site-packages (from seaborn) (2.2.6)',
 'Requirement already satisfied: pandas>=1.2 in /home/ubuntu/.pyenv/versions/3.11.13/lib/python3.11/site-packages (from seaborn) (2.2.3)',
 'Requirement already satisfied: matplotlib!=3.6.1,>=3.4 in /home/ubuntu/.pyenv/versions/3.11.13/lib/python3.11/site-packages (from seaborn) (3.11.1)',
 'Requirement already satisfied: contourpy>=1.0.1 in /home/ubuntu/.pyenv/versions/3.11.13/lib/python3.11/site-packages (from matplotlib!=3.6.1,>=3.4->seaborn) (1.3.3)',
 'Requirement already satisfied: cycler>=0.10 in /home/ubuntu/.pyenv/versions/3.11.13/lib/python3.11/site-packages (from matplotlib!=3.6.1,>=3.4->seaborn) (0.12.1)',
 'Requirement already satisfied: fonttools>=4.28.2 in /home/ubuntu/.pyenv/versions/3.11.13/lib/python3.11

In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import requests
import json

In [4]:
def parse_api(start_date, end_date, hour=None):
    try:
        url = (
            f"https://ap.elementsenergies.com/api/fetchHConsWAvg"
            f"?startdate={start_date}&enddate={end_date}&msn=67001163"
        )
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        print(data)
        records = []

        if isinstance(data["data"], dict):
            for date, hours in data["data"].items():
                for row in hours:
                    timestamp = pd.to_datetime(
                        f"{date} {row['hour']}",
                        format="%Y-%m-%d %H:%M"
                    )

                    records.append({
                        "timestamp": timestamp,
                        "consumption": float(row["consumption"])
                    })

        elif isinstance(data["data"], list):
            date = start_date

            for row in data["data"]:
                timestamp = pd.to_datetime(
                    f"{date} {row['hour']}",
                    format="%Y-%m-%d %H:%M"
                )

                records.append({
                    "timestamp": timestamp,
                    "consumption": float(row["consumption"])
                })

        else:
            raise ValueError("Unexpected API response format.")

        df = pd.DataFrame(records)

        if df.empty:
            return df

        df = df.sort_values("timestamp").reset_index(drop=True)

        if hour is not None:
            df = df[df["timestamp"].dt.hour == hour]
            end_timestamp = pd.to_datetime(end_date).date()
            df = df[
                ~(
                    (df["timestamp"].dt.date == end_timestamp) &
            (df["timestamp"].dt.hour == hour)
            )
            ].reset_index(drop=True)

        return df

    except requests.exceptions.RequestException as e:
        raise Exception(f"API request failed: {e}")

    except KeyError as e:
        raise Exception(f"Missing expected key in API response: {e}")

    except Exception as e:
        raise Exception(f"Unable to parse API response: {e}")

In [5]:
!! lspci | grep -i nvidia

['00:1e.0 3D controller: NVIDIA Corporation TU104GL [Tesla T4] (rev a1)']

In [6]:
data = parse_api("2025-05-16","2026-07-01")

{'scno': 'VSP1664', 'category': 'INDUSTRY (GENERAL)-HT', 'data': {'2025-05-16': [{'hour': '00:00', 'consumption': '0.0'}, {'hour': '01:00', 'consumption': '0.0'}, {'hour': '02:00', 'consumption': '0.0'}, {'hour': '03:00', 'consumption': '0.0'}, {'hour': '04:00', 'consumption': '0.0'}, {'hour': '05:00', 'consumption': '0.0'}, {'hour': '06:00', 'consumption': '0.0'}, {'hour': '07:00', 'consumption': '0.0'}, {'hour': '08:00', 'consumption': '0.0'}, {'hour': '09:00', 'consumption': '0.0'}, {'hour': '10:00', 'consumption': '0.0'}, {'hour': '11:00', 'consumption': '0.0'}, {'hour': '12:00', 'consumption': '0.0'}, {'hour': '13:00', 'consumption': '0.0'}, {'hour': '14:00', 'consumption': '0.0'}, {'hour': '15:00', 'consumption': '50.6'}, {'hour': '16:00', 'consumption': '87.3'}, {'hour': '17:00', 'consumption': '75.1'}, {'hour': '18:00', 'consumption': '65.0'}, {'hour': '19:00', 'consumption': '65.4'}, {'hour': '20:00', 'consumption': '64.6'}, {'hour': '21:00', 'consumption': '63.9'}, {'hour': '

In [7]:
print(data.head(10))

            timestamp  consumption
0 2025-05-16 00:00:00          0.0
1 2025-05-16 01:00:00          0.0
2 2025-05-16 02:00:00          0.0
3 2025-05-16 03:00:00          0.0
4 2025-05-16 04:00:00          0.0
5 2025-05-16 05:00:00          0.0
6 2025-05-16 06:00:00          0.0
7 2025-05-16 07:00:00          0.0
8 2025-05-16 08:00:00          0.0
9 2025-05-16 09:00:00          0.0
